# A spun-coin movie of your own galaxy

This makes a three-panel animation of a galaxy: **surface density**, **line-of-sight velocity**
and **temperature**, with the camera tipping from face-on onto its edge, turning in place,
tumbling in three dimensions the way a spun coin does, and falling flat again. The loop is
seamless, so it repeats forever without a jump.

**To run it on your own simulation, change two things in the next cell**: the path and the output
number. Nothing else is specific to the galaxy used here.

Every frame is a real off-axis projection. Nothing is interpolated between frames and nothing is
masked or cut, so what you see is the data at that viewing angle.

> This is a gallery recipe. It is not part of Mera's test suite, and it is not executed when the
> documentation is built, so its outputs are not stored here. Run it and it will produce them.

**What you need:** Mera, CairoMakie, ColorSchemes, and `ffmpeg` on your PATH.
A run of about 50 million cells needs roughly 9 GB of memory and takes about a minute per frame
on eight threads. Start with `NFRAMES = 12` to see a short loop before committing to the full one.

![The finished animation](media/coin_flip_preview.gif)

*Preview, reduced. The full-resolution video is [`media/coin_flip_movie.mp4`](media/coin_flip_movie.mp4).*


| | |
|---|---|
| **Author** | Manuel Behrendt, LMU Munich |
| **Contact** | [github.com/ManuelBehrendt](https://github.com/ManuelBehrendt) |
| **Reads** | RAMSES |
| **Provenance** | `Mera v1.8.0 \| AV05CD/output_00390 \| 445.9 Myr \| L=48.0 ndim=3 lmin=6 lmax=12` |
| **Status** | checked 2026-09-11 |


## The environment

This recipe carries its own `Project.toml` and `Manifest.toml`. Running the cell below uses exactly
the package versions the recipe was written against, whatever else you have installed. That, plus
the provenance line in the table above, is what makes the result reproducible: the provenance says
which Mera and which snapshot, the manifest says which versions of everything else.

`instantiate` downloads those versions the first time and does nothing on later runs. See
[Reproducibility](https://manuelbehrendt.github.io/Mera.jl/stable/reproducibility/).


In [ ]:
using Pkg
Pkg.activate(@__DIR__)        # the Project.toml and Manifest.toml next to this notebook
Pkg.instantiate()             # first run only: fetches exactly those versions


## 1. The two lines you change

In [ ]:
using Mera, Printf, ColorSchemes
using CairoMakie
CairoMakie.activate!(type = "png")

# ---- change these two ----------------------------------------------------------------
const SIM    = "/path/to/your/simulation"     # a RAMSES output folder, or a MERA-file folder
const OUTPUT = 390                            # the snapshot number
# --------------------------------------------------------------------------------------

const NFRAMES   = 180        # 9 seconds at 20 fps; try 12 first
const R_SELECT  = 26.0       # kpc, how much of the box to take in
const FRAME     = 16.0       # kpc, half width of what is shown
const PXSIZE_PC = 90.0       # pixel size; smaller is sharper and slower
const TILT_HI   = 87.0       # the steepest tilt, just short of exactly edge-on
const AZ_MEAN   = 35.0       # a starting azimuth that is not axis aligned

const OUTDIR = "coin_frames"
const GIFOUT = "galaxy_coin.gif"
const MP4OUT = "galaxy_coin.mp4"
const FFMPEG = "ffmpeg"      # or an absolute path

## 2. Load the snapshot

`loadall` reads whatever the snapshot holds in one call and works the same on a raw RAMSES output
and on a converted MERA file, so the line does not change when you convert your data. Here only
the gas is needed.

In [ ]:
# loadall reads every component the snapshot holds and returns a NamedTuple, so `info` comes
# back too. It dispatches on what it finds: a folder of RAMSES files is read with the RAMSES
# readers, a folder holding output_NNNNN.jld2 with `loaddata`. The line is the same either way,
# which is why converting your data does not mean rewriting this notebook.
(; hydro, info) = loadall(SIM, OUTPUT)

FINEST_PC = round(info.boxlen * info.scale.pc / 2^info.levelmax, digits = 1)
@printf("%d cells, finest cell %.1f pc\n", length(hydro.data), FINEST_PC)

## 3. The camera window: use `fov`, not a box

This is the first thing that goes wrong if you improvise it.

`xrange`/`yrange`/`zrange` are **world-space** bounds. A cubic window is not rotation invariant, so
as the camera turns, the window's own flat faces project into the picture as straight edges, and
you get a rotating rectangle across your galaxy.

`fov` asks for a camera-plane window instead, taken from a sphere, and a sphere projects to the
same disc at every orientation. `aperture = :circle` keeps that disc. Selecting out to `R_SELECT`
while framing at the smaller `FRAME` keeps even the sphere's own edge outside the picture.

In [ ]:
# Every projection in this notebook takes these. Three angles are what make the motion read as
# three-dimensional: `inclination` tips the disc towards you, `azimuth` walks the camera around
# it, and `position_angle` rolls the image in its own plane.
view_kw(inc, az, roll) = (inclination = inc, azimuth = az, position_angle = roll,
                          pxsize = [PXSIZE_PC, :pc],
                          fov = R_SELECT, fov_unit = :kpc, aperture = :circle,
                          center = [:bc], verbose = false, show_progress = false)

## 4. The choreography

Four movements, all with smoothstep easing so there are no jerks at the joins:

| fraction of the loop | what happens |
|---|---|
| 0.00 – 0.20 | tips up from face-on onto its edge |
| 0.20 – 0.42 | stays edge-on and turns a full circle in place |
| 0.42 – 0.82 | tumbles in three dimensions, tilt swinging, azimuth running on, image rolling |
| 0.82 – 1.00 | falls flat again |

**Why it loops.** The azimuth advances 1440° and the roll 720°, both whole numbers of turns, so the
last frame reproduces the first. If you change the timings, keep both totals multiples of 360 or
the loop will jump.

Three angles are what make it read as three-dimensional: `inclination` tips the disc,
`azimuth` walks the camera around it, and `position_angle` rolls the image.

In [ ]:
# Smoothstep: the standard ease-in-ease-out curve, 3t^2 - 2t^3 on 0..1.
#
# Why not just use t? A linear ramp starts and stops abruptly: the disc would jerk into motion and
# stop dead. Smoothstep has zero slope at both ends and its steepest slope in the middle, so a
# movement accelerates out of rest and decelerates back into it. That is what makes the joins
# between the four movements below invisible.
#
#   t          0     0.25    0.5    0.75     1
#   t          0.00  0.25    0.50   0.75  1.00   <- linear, constant speed
#   smoothstep 0.00  0.16    0.50   0.84  1.00   <- slow, fast, slow
#
# The clamps matter: called with t outside 0..1 it would otherwise run away, and each movement
# below hands it a fraction of its own span.
smoothstep(t) = t <= 0 ? 0.0 :          # before the movement starts: hold at the beginning
                t >= 1 ? 1.0 :          # after it ends: hold at the end
                t * t * (3 - 2t)        # in between: ease in, then ease out

"""Camera for frame k, as (inclination, azimuth, roll) in degrees.

Each branch maps its own slice of the loop onto 0..1 and eases across it, so the four movements
join without a visible change of speed.
"""
function camera(k)
    f = k / NFRAMES                               # position in the loop, 0 at the start, 1 at the end

    if f < 0.20                                   # --- tip up onto the edge -------------------
        # inclination 0 -> TILT_HI; azimuth and roll stay put, so the disc simply leans back
        (TILT_HI * smoothstep(f / 0.20), AZ_MEAN, 0.0)

    elseif f < 0.42                               # --- turn in place, edge-on ----------------
        u = (f - 0.20) / 0.22                     # 0..1 across this movement
        # one full turn of azimuth at fixed tilt: the camera walks around the edge-on disc
        (TILT_HI, AZ_MEAN + 360.0 * smoothstep(u), 0.0)

    elseif f < 0.82                               # --- tumble in three dimensions ------------
        u   = (f - 0.42) / 0.40
        wob = sin(2π * 1.5 * u)                   # one and a half swings of the tilt
        # abs() makes the tilt dip and return rather than pass through face-on twice; the
        # (1 - 0.35u) factor shrinks each swing a little, so the tumble calms as it goes
        (TILT_HI - 46.0 * abs(wob) * (1 - 0.35u),
         AZ_MEAN + 360.0 + 720.0 * smoothstep(u),   # two more turns of azimuth
         360.0 * smoothstep(u) + 90.0 * sin(2π * 2 * u))   # a full roll, plus a wobble on top

    else                                          # --- fall flat -----------------------------
        u = smoothstep((f - 0.82) / 0.18)
        # inclination back to 0 while azimuth and roll finish their last turn
        (TILT_HI * (1 - u), AZ_MEAN + 1080.0 + 360.0 * u, 360.0 + 360.0 * u)
    end
end

# Check the loop closes BEFORE rendering. Cheap, and it saves discovering a jump after an hour
# of rendering: the first and last frame must agree in all three angles, modulo whole turns.
let (i0, a0, r0) = camera(0), (iN, aN, rN) = camera(NFRAMES)
    @printf("start %.1f %.1f %.1f   end %.1f %.1f %.1f   closes: %s\n",
            i0, a0, r0, iN, mod(aN, 360), mod(rN, 360),
            abs(i0 - iN) < 0.01 && abs(mod(aN - a0, 360)) < 0.01 && abs(mod(rN - r0, 360)) < 0.01)
end

## 5. The three panels, and why these colour maps

| panel | quantity | map | why |
|---|---|---|---|
| Σ | surface density | `magma` | perceptually uniform, monotonic lightness |
| v_LOS | line-of-sight velocity | `berlin` | **diverging with a dark centre** |
| T | temperature | `lajolla` | sequential, distinct from magma |

All three stay readable with colour vision deficiency: `magma` survives being seen in greyscale,
and `berlin` and `lajolla` are Crameri scientific colour maps built for exactly this.

**Fix the limits once, for the whole movie.** Recomputing them per frame makes the brightness pulse
as the camera turns, which is the single thing that makes a rotation movie look amateurish. Adjust
these to your own galaxy: run one frame, look, adjust.

In [ ]:
# quantity, unit, label, colormap, (lo, hi), log?
const PANELS = [
    (:sd,   :Msol_pc2, "Σ",     ColorSchemes.magma,   (-3.0,   2.5), true),
    (:vlos, :km_s,     "v_LOS", ColorSchemes.berlin,  (-200.0, 200.0), false),
    (:T,    :K,        "T",     ColorSchemes.lajolla, (3.5,    6.5), true),
]
const UNITS = ["log₁₀ M⊙/pc²", "km/s", "log₁₀ K"]
const TICKS = [[-2, 0, 2], [-150, 0, 150], [4, 5, 6]]

## 6. One fixed grid for every frame

The second thing that goes wrong. The projected extent **changes with viewing angle**, because the
corners of the box rotate into view: in the run this was written for, the frame spans 15.2 kpc at
one azimuth and 21.5 kpc at another, a 41% swing. Left alone, the galaxy changes size from frame to
frame, and since the figure height follows the map, so does the text. The movie visibly pulses.

Resampling every frame onto one grid fixed for the whole sequence removes it.

In [ ]:
# Map physical values onto 0..1 for the colour scale. Two cases: log for quantities that span
# decades (surface density, temperature), linear for one that does not and is signed (velocity).
# NaN is checked first, because NaN is also not finite and would otherwise be caught below and
# turned into 0, which would paint empty sky as the bottom of the colour scale instead of leaving
# it as background.
function stretch(a, lo, hi, uselog)
    out = similar(a, Float64)
    @inbounds for i in eachindex(a)
        v = a[i]
        out[i] = isnan(v)      ? NaN :
                 !isfinite(v)  ? 0.0 :
                 uselog        ? (v > 0 ? (log10(v) - lo) / (hi - lo) : 0.0) :
                                 (v - lo) / (hi - lo)
    end
    return clamp.(out, 0.0, 1.0)
end

"""
Resample a map onto a fixed physical grid, matching by coordinate rather than by index.

Nearest-neighbour is enough here: the target grid is close to the source resolution, so
interpolating would only blur it. Pixels of the target that fall outside the source stay NaN and
are drawn as background.
"""
function regrid(m, ext_src, nx, ny, ext_dst)
    out = fill(NaN, nx, ny)
    sx, sy = size(m)
    x0s, x1s, y0s, y1s = ext_src
    x0d, x1d, y0d, y1d = ext_dst
    @inbounds for j in 1:ny, i in 1:nx
        x = x0d + (i - 0.5) * (x1d - x0d) / nx
        y = y0d + (j - 0.5) * (y1d - y0d) / ny
        (x < x0s || x > x1s || y < y0s || y > y1s) && continue
        is = clamp(ceil(Int, (x - x0s) / (x1s - x0s) * sx), 1, sx)
        js = clamp(ceil(Int, (y - y0s) / (y1s - y0s) * sy), 1, sy)
        out[i, j] = m[is, js]
    end
    return out
end

# The grid every frame is resampled onto: a square FRAME kpc across, at PXSIZE_PC per pixel.
# Fixed once, outside the loop, which is the whole point.
const TARGET = (ext = [-FRAME, FRAME, -FRAME, FRAME],
                nx  = round(Int, 2FRAME * 1000 / PXSIZE_PC),
                ny  = round(Int, 2FRAME * 1000 / PXSIZE_PC))

## 7. Render the frames

One projection per panel per frame. Colourbars sit under their panels, narrower than the panel and
with three explicit ticks, because the default tick set runs labels to the very ends where
neighbouring bars collide.

In [ ]:
mkpath(OUTDIR)
t0 = time()
for k in 0:(NFRAMES - 1)
    inc, az, roll = camera(k)                     # where the camera is for this frame
    kw = view_kw(inc, az, roll)

    # One real projection per panel. This is the expensive part: each call walks every cell in
    # the snapshot and deposits it onto the camera plane, which is why nothing here is
    # interpolated between frames.
    imgs = Matrix{Float64}[]
    for (q, u, _, _, lims, uselog) in PANELS
        pr = projection(hydro, q, u; kw...)
        # stretch with the FIXED limits, then resample onto the FIXED grid: the two things that
        # keep the movie from pulsing in brightness and in size
        push!(imgs, regrid(stretch(pr.maps[q], lims[1], lims[2], uselog),
                           pr.extent, TARGET.nx, TARGET.ny, TARGET.ext))
    end

    # Three panels side by side on a black canvas, sized from the grid so the aspect is right
    ext = TARGET.ext
    nx, ny = size(imgs[1]); panel_w = 500
    panel_h = round(Int, panel_w * ny / nx)
    fig = Figure(size = (3panel_w, panel_h + 76), backgroundcolor = :black,
                 figure_padding = (10, 10, 10, 6))

    for (n, (q, _, label, cs, lims, _)) in enumerate(PANELS)
        ax = Axis(fig[1, n]; aspect = DataAspect(), backgroundcolor = :black)
        hidedecorations!(ax); hidespines!(ax)
        # colorrange is 0..1 because `stretch` already mapped the physical values onto that;
        # the colourbar below restores the real numbers. nan_color paints untouched pixels as
        # background rather than as the bottom of the scale.
        image!(ax, ext[1] .. ext[2], ext[3] .. ext[4], imgs[n];
               colormap = cgrad(cs), colorrange = (0.0, 1.0), nan_color = :black)
        text!(ax, ext[1] + 0.7, ext[4] - 0.5; text = label, color = :white,
              fontsize = 21, font = :bold, align = (:left, :top))
        Colorbar(fig[2, n]; colormap = cgrad(cs), limits = lims, vertical = false,
                 flipaxis = false, height = 9, width = Relative(0.72), ticks = TICKS[n],
                 ticklabelsize = 11, ticklabelpad = 3,
                 ticklabelcolor = RGBf(0.72, 0.74, 0.80), tickcolor = RGBf(0.5, 0.5, 0.55),
                 label = UNITS[n], labelsize = 11, labelcolor = RGBf(0.55, 0.57, 0.63),
                 labelpadding = 2, spinewidth = 0)
    end
    rowgap!(fig.layout, 8); colgap!(fig.layout, 16)

    save(joinpath(OUTDIR, @sprintf("coin_%03d.png", k)), fig)
    el = time() - t0
    @printf("  frame %3d/%d  inc=%5.1f  %5.1f s, ~%4.1f min left\n",
            k + 1, NFRAMES, inc, el, (el / (k + 1) * (NFRAMES - k - 1)) / 60)
end

## 8. Assemble

The mp4 is roughly a tenth the size of the gif at better quality and is what a web page wants. The
gif is for places that will not play video, such as a GitHub README.

In [ ]:
# ffmpeg reads the numbered frames as a sequence. -loop 0 makes the gif repeat forever; the
# even-height constraint (-2) matters because h264 requires it.
pat = joinpath(OUTDIR, "coin_%03d.png")
run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat -vf "scale=1200:-2:flags=lanczos" -loop 0 $GIFOUT`)
run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat -vf "scale=1500:-2:flags=lanczos"
     -c:v libx264 -pix_fmt yuv420p -crf 20 -movflags +faststart $MP4OUT`)
@printf("gif %.1f MB, mp4 %.1f MB\n", filesize(GIFOUT)/1e6, filesize(MP4OUT)/1e6)

## The same thing, short

Everything above spelled out is what you need in order to *change* it. Once the choices are made,
Mera's own shorthands carry most of it. `withargs` derives a per-frame variant of one argument
bundle without touching the original, and `@project` projects all three quantities in a single line
and binds each map to a variable of its own, so there is no `proj.maps[...]` indexing.

The parts that cannot be shortened are the parts that are decisions rather than plumbing: the
choreography, the fixed colour limits, and the fixed grid.

In [ ]:
using Mera, CairoMakie, ColorSchemes, Printf

(; hydro) = loadall(SIM, OUTPUT)

base = ArgumentsType(fov = R_SELECT, fov_unit = :kpc, aperture = :circle,
                     center = [:bc], pxsize = [PXSIZE_PC, :pc],
                     verbose = false, show_progress = false)

maps3 = [ColorSchemes.magma, ColorSchemes.berlin, ColorSchemes.lajolla]
lims3 = [(-3.0, 2.5), (-200.0, 200.0), (3.5, 6.5)]

mkpath(OUTDIR)
for k in 0:(NFRAMES - 1)
    inc, az, roll = camera(k)
    args = withargs(base; inclination = inc, azimuth = az, position_angle = roll)

    @project hydro sd=>:Msol_pc2 vlos=>:km_s T=>:K myargs=args

    fig = Figure(size = (1500, 560), backgroundcolor = :black)
    for (n, (m, cs, lims, lg)) in enumerate(zip((sd, vlos, T), maps3, lims3, (true, false, true)))
        ax = Axis(fig[1, n]; aspect = DataAspect(), backgroundcolor = :black)
        hidedecorations!(ax); hidespines!(ax)
        img = regrid(stretch(m, lims[1], lims[2], lg), proj.extent,
                     TARGET.nx, TARGET.ny, TARGET.ext)
        image!(ax, TARGET.ext[1] .. TARGET.ext[2], TARGET.ext[3] .. TARGET.ext[4], img;
               colormap = cgrad(cs), colorrange = (0, 1), nan_color = :black)
    end
    save(joinpath(OUTDIR, @sprintf("coin_%03d.png", k)), fig)
end

## Does threading make it faster?

Yes, and most of it is already happening.

**Each projection is already multithreaded.** Mera spreads the deposit of cells onto the camera
plane across all available threads, so the single biggest thing you can do is start Julia with
threads at all:

```bash
julia -t 8 --project    # or JULIA_NUM_THREADS=8
```

A single-threaded run of this notebook is several times slower for no other reason.

**Frames run one after another here, on purpose.** Mera can invert that: `rotation_sequence` takes
`parallel_frames=true`, which runs the *frames* concurrently with each projection single-threaded.
Its documentation puts that at roughly 1.5 to 2 times faster for an orbit, because it fills every
core when there are more frames than threads, at the cost of proportionally more transient memory.

That option does not apply to this notebook, because `rotation_sequence` sweeps **one** angle,
`:azimuth`, `:inclination` or `:position_angle`, while the tumble here moves all three at once. If
your animation is a simple orbit, use it and skip the hand-written loop entirely:

```julia
maps = rotation_sequence(hydro, :sd, :Msol_pc2;
                         sweep = :azimuth, angles = range(0, 360, length = 120),
                         inclination = 70, fov = 26, fov_unit = :kpc,
                         parallel_frames = true)
```

**Threading the loop yourself is not worth it.** You could wrap the projections in
`Threads.@threads`, but the figure writing is the part that does not parallelise safely, and the
projections are already using the cores. You would be competing with yourself.

**What actually helps most, in order:**

1. **Fewer cells.** Cost scales with the number of cells the projection walks, not with pixels.
   `loadall(SIM, OUTPUT; lmax = 11)` on a RAMSES output, or a spatial selection, cuts the work
   directly. Halving the cells roughly halves the time.
2. **Fewer frames while you iterate.** `NFRAMES = 12` renders a rough loop in a minute or two.
   Settle the colour limits and the choreography there, then run the long one once.
3. **Larger pixels.** `PXSIZE_PC` is a second-order effect here but still real.
4. **Fewer panels.** Three quantities means three projections per frame. Dropping to one while you
   tune the motion makes iteration three times faster.

## Making it yours

- **Different quantities.** Any three `getvar` can compute. `:σlos` for the line-of-sight
  dispersion instead of `:T` shows a cold disc inside a turbulent envelope. Remember that a signed
  quantity wants a diverging map and an unsigned one a sequential map.
- **A single phase.** `filterdata(hydro, Below(:T, 300.0, :K), Above(:rho, 75.0, :nH))` gives the
  molecular gas as a chainable object you can project exactly like `hydro`.
- **A plain orbit.** Replace `camera` with a constant inclination and an azimuth running
  0 to 360. Simpler, and it still loops.
- **Sharper.** Lower `PXSIZE_PC`. Cost grows roughly as its square.
- **Too slow to iterate?** Set `NFRAMES = 12` and look at one frame before running the full loop.

If you build something with this, a link in
[Discussions](https://github.com/ManuelBehrendt/Mera.jl/discussions) is welcome.